# Stage 7: Test Inference, Output Formatting & Submission Validation

## Install Dependencies & Download Submission Validator

In [1]:
!pip uninstall -y -q datasets
!pip install -q -U s3fs boto3 botocore pyarrow lightgbm scikit-learn rapidfuzz tqdm
print("✅ Runtime equipped with high-throughput inference drivers!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.0/102.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.7/221.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sourc

## Credentials, Directory Setup & Locked Parameters

In [1]:
import os
import gc
import re
import csv
import time
import joblib
import unicodedata
from collections import Counter, defaultdict
from typing import Any, Dict, List, Set, Tuple
import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from google.colab import userdata
import s3fs

# 1. AWS Credentials
AWS_ACCESS_KEY_ID = userdata.get("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = userdata.get("AWS_SECRET_ACCESS_KEY")
AWS_DEFAULT_REGION = userdata.get("AWS_DEFAULT_REGION") or "us-east-1"
S3_BUCKET = userdata.get("S3_BUCKET_NAME").replace("s3://", "").strip("/")

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_DEFAULT_REGION"] = AWS_DEFAULT_REGION

s3_storage_options = {
    "key": AWS_ACCESS_KEY_ID,
    "secret": AWS_SECRET_ACCESS_KEY,
    "client_kwargs": {"region_name": AWS_DEFAULT_REGION}
}

# 2. Paths
TEST_BASE_S3 = f"s3://{S3_BUCKET}/student_resource/dataset/test"
MODEL_S3_KEY = f"{S3_BUCKET}/models/lgbm_model_latest.pkl"
SUBMISSIONS_S3 = f"s3://{S3_BUCKET}/submissions"

# Local output directory for submission
os.makedirs("output", exist_ok=True)
MATCHING_TSV = "output/matching_results.tsv"
CANDIDATE_TSV = "output/candidate_pairs.tsv"

# 3. Locked Decision Threshold (Tuned in Stage 6)
OPTIMAL_THRESHOLD = 0.74

print(f"✅ S3 Credentials verified.")
print(f"📁 Test S3 Base     : {TEST_BASE_S3}")
print(f"🤖 Model Checkpoint : s3://{MODEL_S3_KEY}")
print(f"🎯 Locked Threshold : {OPTIMAL_THRESHOLD}")
print(f"📄 Output Files     : {MATCHING_TSV}, {CANDIDATE_TSV}")

✅ S3 Credentials verified.
📁 Test S3 Base     : s3://amz-ml-crazy-dave-bucket-177683310295-us-east-1-an/student_resource/dataset/test
🤖 Model Checkpoint : s3://amz-ml-crazy-dave-bucket-177683310295-us-east-1-an/models/lgbm_model_latest.pkl
🎯 Locked Threshold : 0.74
📄 Output Files     : output/matching_results.tsv, output/candidate_pairs.tsv


## Load LightGBM Model & Production Normalizer

In [2]:
fs = s3fs.S3FileSystem(**s3_storage_options)
local_model = "/tmp/lgbm_model_latest.pkl"
fs.get(MODEL_S3_KEY, local_model)
model = joblib.load(local_model)
print(f"✅ Model loaded into memory (Best Iteration: {model.best_iteration_}).")

# Fast C++ Normalization & Tokenization Engine
RE_COMBINING = re.compile(r"[̀-ͯ]")
RE_ALLOWED = re.compile(r"[^\w\sऀ-ॿ]+", re.UNICODE)
RE_SPACES = re.compile(r"\s+")
RE_POSTAL = re.compile(r"\d{5,6}")
RE_PVT_LTD = re.compile(r"(pvt\.?\s*ltd\.?|private\s+limited|p\.?\s*ltd\.?)", re.IGNORECASE)
RE_INC = re.compile(r"(corporation|corp\.?|incorporated|inc\.?)", re.IGNORECASE)
RE_LTD = re.compile(r"(limited|ltd\.?)", re.IGNORECASE)
RE_LLC = re.compile(r"(llp|llc)", re.IGNORECASE)

# High-frequency generic business & domain stopwords
NAME_STOPWORDS: Set[str] = {
    "and", "the", "for", "with", "ltd", "pvt", "inc", "corp", "llc", "llp",
    "limited", "private", "corporation", "company", "co", "enterprises",
    "services", "solutions", "international", "group", "technologies",
    "industries", "trading", "associates", "consulting", "holdings",
    "de", "la", "le", "les", "et", "du", "des", "sarl", "sa", "sas", "societe",
    "india", "usa", "us", "france", "fr",
    "hotel", "restaurant", "store", "shop", "mart", "agency", "travels"
}

# High-frequency address & street noise stopwords
ADDR_STOPWORDS: Set[str] = NAME_STOPWORDS | {
    "road", "street", "st", "rd", "lane", "ave", "avenue", "blvd", "boulevard",
    "floor", "bldg", "building", "near", "opp", "opposite", "behind", "beside",
    "post", "dist", "district", "city", "state", "nagar", "colony", "chowk",
    "marg", "cross", "main", "layout", "phase", "sector", "block", "plot",
    "no", "house", "flat", "apartment", "complex", "plaza", "tower", "towers",
    "rue", "passage", "allee", "route", "zone", "industrial", "area"
}

def clean_str(text: Any) -> str:
    if not text or not isinstance(text, str) or str(text).lower() == "nan":
        return ""
    text = unicodedata.normalize("NFKD", text)
    text = RE_COMBINING.sub("", text).lower()
    text = RE_PVT_LTD.sub("pvt ltd", text)
    text = RE_INC.sub("inc", text)
    text = RE_LTD.sub("ltd", text)
    text = RE_LLC.sub("llc", text)
    text = RE_ALLOWED.sub(" ", text)
    return RE_SPACES.sub(" ", text).strip()

def extract_name_tokens(text: str) -> List[str]:
    return [t for t in text.split() if len(t) >= 3 and t not in NAME_STOPWORDS and not t.isdigit()]

def extract_addr_tokens(text: str) -> List[str]:
    return [t for t in text.split() if len(t) >= 3 and t not in ADDR_STOPWORDS and not t.isdigit()]

def extract_postal_code(text: str) -> Optional[str]:
    m = RE_POSTAL.search(text)
    return f"PIN_{m.group(0)}" if m else None

FEATURE_COLS = [
    "name_ratio", "name_partial", "name_token_sort", "name_token_set",
    "addr_ratio", "addr_partial", "addr_token_set", "name_len_diff", "addr_len_diff"
]


✅ Model loaded into memory (Best Iteration: 254).


## Load Test S2 & S3 and Build Partitioned Inverted Indexes

In [3]:
print("🏗️ Building High-Recall Dual Inverted Indexes (Name + Postal + Address)...")
t_index_start = time.perf_counter()

REQUIRED_COLS = ["entity_id", "business_name", "business_address", "country"]

# Country-partitioned isolated indexes
country_name_inv = defaultdict(lambda: defaultdict(list))
country_postal_inv = defaultdict(lambda: defaultdict(list))
country_addr_inv = defaultdict(lambda: defaultdict(list))
country_entities = defaultdict(lambda: ([], [], []))

total_ref_entities = 0
CAP_PER_SOURCE = 1_500  # Fair 1,500 slots per source guarantees Source 3 is NEVER locked out!

for src_file in ["test_source2.tsv", "test_source3.tsv"]:
    print(f"⏳ Streaming {src_file} from S3...")
    t_file = time.perf_counter()

    df_part = pd.read_csv(
        f"{TEST_BASE_S3}/{src_file}",
        sep="	",
        dtype=str,
        usecols=lambda c: c in REQUIRED_COLS,
        storage_options=s3_storage_options
    )
    df_part["country"] = df_part["country"].fillna("UNKNOWN").str.upper().str.strip()

    for c in df_part["country"].unique():
        sub = df_part[df_part["country"] == c]
        ids = sub["entity_id"].astype(str).tolist()
        names = [clean_str(x) for x in sub["business_name"].fillna("")]
        addrs = [clean_str(x) for x in sub["business_address"].fillna("")]

        ref_ids, ref_names, ref_addrs = country_entities[c]
        base_offset = len(ref_ids)

        ref_ids.extend(ids)
        ref_names.extend(names)
        ref_addrs.extend(addrs)

        name_inv = country_name_inv[c]
        postal_inv = country_postal_inv[c]
        addr_inv = country_addr_inv[c]

        # Per-source posting counter to ensure equal and fair representation
        src_name_counts = Counter()
        src_addr_counts = Counter()

        for local_idx, (name, addr) in enumerate(zip(names, addrs)):
            global_idx = base_offset + local_idx

            # 1. Pure Name Inverted Index (Uncontaminated by address tokens)
            name_toks = set(extract_name_tokens(name))
            for tok in name_toks:
                if src_name_counts[tok] < CAP_PER_SOURCE:
                    name_inv[tok].append(global_idx)
                    src_name_counts[tok] += 1

            # 2. Dedicated Postal Code Inverted Index (Exact PIN/ZIP bucket)
            pin = extract_postal_code(addr)
            if pin:
                postal_inv[pin].append(global_idx)

            # 3. Discriminative Address Inverted Index
            addr_toks = set(extract_addr_tokens(addr))
            for tok in addr_toks:
                if src_addr_counts[tok] < CAP_PER_SOURCE:
                    addr_inv[tok].append(global_idx)
                    src_addr_counts[tok] += 1

    total_ref_entities += len(df_part)
    print(f"  ↳ Indexed {src_file} ({len(df_part):,} records) in {time.perf_counter() - t_file:.1f}s")
    del df_part
    gc.collect()

print(f"
🎉 Indexed {total_ref_entities:,} total reference entities across {len(country_name_inv)} countries in {time.perf_counter() - t_index_start:.1f}s!")
for c in sorted(country_name_inv.keys()):
    ids, _, _ = country_entities[c]
    print(f"  • Country [{c}]: {len(ids):,} entities | {len(country_name_inv[c]):,} Name Tokens | {len(country_postal_inv[c]):,} Postal Codes | {len(country_addr_inv[c]):,} Addr Tokens")


🏗️ Building High-Recall Inverted Indexes (Name + Address Tokens)...
⏳ Streaming test_source2.tsv from S3...
  ↳ Indexed test_source2.tsv (4,887,273 records) in 183.5s
⏳ Streaming test_source3.tsv from S3...
  ↳ Indexed test_source3.tsv (5,082,316 records) in 181.6s

🎉 Indexed 9,969,589 total reference entities across 3 countries in 370.1s!
  • Country [FRANCE]: 1,434,993 entities | 232,794 unique tokens.
  • Country [INDIA]: 4,717,565 entities | 907,838 unique tokens.
  • Country [US]: 3,817,031 entities | 814,117 unique tokens.


## High-Speed Streaming Inference

In [8]:
CHUNK_SIZE = 150_000

# Initialize TSV files with exact headers
with open(MATCHING_TSV, "w", encoding="utf-8") as f_match,      open(CANDIDATE_TSV, "w", encoding="utf-8") as f_cand:
    f_match.write("source1_entity_id	matched_entity_ids
")
    f_cand.write("source1_entity_id	candidate_entity_ids
")

print(f"🚀 Streaming test_source1.tsv from S3 in batches of {CHUNK_SIZE:,} ...")
t_start = time.perf_counter()
total_processed = 0
total_matches_emitted = 0

s1_reader = pd.read_csv(
    f"{TEST_BASE_S3}/test_source1.tsv",
    sep="	",
    dtype=str,
    chunksize=CHUNK_SIZE,
    storage_options=s3_storage_options
)

for chunk_idx, chunk in enumerate(s1_reader, 1):
    t_chunk = time.perf_counter()
    chunk["country"] = chunk["country"].fillna("UNKNOWN").str.upper().str.strip()

    s1_ids = chunk["entity_id"].astype(str).tolist()
    s1_names = [clean_str(x) for x in chunk["business_name"].fillna("")]
    s1_addrs = [clean_str(x) for x in chunk["business_address"].fillna("")]
    s1_countries = chunk["country"].tolist()

    chunk_candidates = defaultdict(list)
    pair_rows = []

    # High-Recall & C-Accelerated Set Intersection Candidate Retrieval
    for i in range(len(s1_ids)):
        s1_id = s1_ids[i]
        c = s1_countries[i]
        n_clean = s1_names[i]
        a_clean = s1_addrs[i]

        name_tokens = extract_name_tokens(n_clean)
        pin = extract_postal_code(a_clean)
        addr_tokens = extract_addr_tokens(a_clean)

        cand_indices = []
        seen_cand = set()

        if c in country_name_inv and name_tokens:
            name_inv = country_name_inv[c]
            postal_inv = country_postal_inv[c]
            addr_inv = country_addr_inv[c]
            ref_ids, ref_names, ref_addrs = country_entities[c]

            # Filter tokens and sort by rarity (shortest posting list first)
            v_name_tokens = [t for t in name_tokens if t in name_inv]

            if v_name_tokens:
                v_name_tokens.sort(key=lambda t: len(name_inv[t]))

                # Strategy 1: Postal Code (PIN) + Rarest Name Token Exact Intersection
                if pin and pin in postal_inv:
                    pin_postings = set(postal_inv[pin])
                    pin_intersect = pin_postings & set(name_inv[v_name_tokens[0]])
                    for idx in pin_intersect:
                        if idx not in seen_cand:
                            seen_cand.add(idx)
                            cand_indices.append(idx)
                            if len(cand_indices) >= 5:
                                break

                # Strategy 2: Multi-Token Intersection of Top 2 Rarest Name Tokens
                if len(cand_indices) < 5 and len(v_name_tokens) >= 2:
                    t0, t1 = v_name_tokens[0], v_name_tokens[1]
                    name_intersect = set(name_inv[t0]) & set(name_inv[t1])
                    for idx in name_intersect:
                        if idx not in seen_cand:
                            seen_cand.add(idx)
                            cand_indices.append(idx)
                            if len(cand_indices) >= 5:
                                break

                # Strategy 3: Locality Guided Fallback (Single-Token Names or Missing Intersections)
                # Intersect rarest name token with rarest discriminative address token
                if len(cand_indices) < 5 and addr_tokens:
                    v_addr_tokens = [t for t in addr_tokens if t in addr_inv]
                    if v_addr_tokens:
                        v_addr_tokens.sort(key=lambda t: len(addr_inv[t]))
                        name_addr_intersect = set(name_inv[v_name_tokens[0]]) & set(addr_inv[v_addr_tokens[0]])
                        for idx in name_addr_intersect:
                            if idx not in seen_cand:
                                seen_cand.add(idx)
                                cand_indices.append(idx)
                                if len(cand_indices) >= 5:
                                    break

                # Strategy 4: High-Precision Rarest Name Token Head (up to 15 candidates)
                if len(cand_indices) < 5:
                    for idx in name_inv[v_name_tokens[0]][:15]:
                        if idx not in seen_cand:
                            seen_cand.add(idx)
                            cand_indices.append(idx)
                            if len(cand_indices) >= 5:
                                break

                # Strategy 5: Second Rarest Name Token Head (if still under 5)
                if len(cand_indices) < 5 and len(v_name_tokens) >= 2:
                    for idx in name_inv[v_name_tokens[1]][:10]:
                        if idx not in seen_cand:
                            seen_cand.add(idx)
                            cand_indices.append(idx)
                            if len(cand_indices) >= 5:
                                break

                for s23_idx in cand_indices:
                    cand_id = ref_ids[s23_idx]
                    chunk_candidates[s1_id].append(cand_id)
                    pair_rows.append((
                        s1_id, cand_id,
                        n_clean, ref_names[s23_idx],
                        a_clean, ref_addrs[s23_idx]
                    ))

    # Vectorized RapidFuzz Feature Calculation in C++
    matched_map = defaultdict(list)
    if pair_rows:
        n_pairs = len(pair_rows)
        X_mat = np.empty((n_pairs, 9), dtype=np.float32)

        for p_idx, (s1_id, c_id, n1, n2, a1, a2) in enumerate(pair_rows):
            X_mat[p_idx, 0] = fuzz.ratio(n1, n2) / 100.0
            X_mat[p_idx, 1] = fuzz.partial_ratio(n1, n2) / 100.0
            X_mat[p_idx, 2] = fuzz.token_sort_ratio(n1, n2) / 100.0
            X_mat[p_idx, 3] = fuzz.token_set_ratio(n1, n2) / 100.0
            X_mat[p_idx, 4] = fuzz.ratio(a1, a2) / 100.0
            X_mat[p_idx, 5] = fuzz.partial_ratio(a1, a2) / 100.0
            X_mat[p_idx, 6] = fuzz.token_set_ratio(a1, a2) / 100.0
            X_mat[p_idx, 7] = abs(len(n1) - len(n2))
            X_mat[p_idx, 8] = abs(len(a1) - len(a2))

        # Model Inference with calibrated threshold
        X_df = pd.DataFrame(X_mat, columns=FEATURE_COLS)
        probs = model.predict_proba(X_df)[:, 1]

        # Precision-First Top-1 Match Selection (Protects Macro F_0.5 from false merge penalty)
        s1_pair_indices = defaultdict(list)
        for p_idx, (s1_id, cand_id, _, _, _, _) in enumerate(pair_rows):
            s1_pair_indices[s1_id].append(p_idx)

        for s1_id, p_indices in s1_pair_indices.items():
            best_p_idx = max(p_indices, key=lambda pi: probs[pi])
            best_prob = probs[best_p_idx]

            if best_prob >= OPTIMAL_THRESHOLD:
                best_cand = pair_rows[best_p_idx][1]
                matches = [best_cand]

                # Rare tie-breaker: Only emit a 2nd match if probability >= 0.92 and within 0.03 of best
                for other_pi in p_indices:
                    if other_pi != best_p_idx and probs[other_pi] >= 0.92 and (best_prob - probs[other_pi]) <= 0.03:
                        matches.append(pair_rows[other_pi][1])

                matched_map[s1_id] = matches

    # Append to TSV files directly
    with open(MATCHING_TSV, "a", encoding="utf-8") as f_match,          open(CANDIDATE_TSV, "a", encoding="utf-8") as f_cand:
        for s1_id in s1_ids:
            cands = chunk_candidates.get(s1_id, [])
            matches = matched_map.get(s1_id, [])

            f_cand.write(f"{s1_id}	{','.join(cands)}
")
            f_match.write(f"{s1_id}	{','.join(matches)}
")
            if matches:
                total_matches_emitted += 1

    total_processed += len(s1_ids)
    batch_time = time.perf_counter() - t_chunk
    speed = len(s1_ids) / batch_time
    print(f"  • Batch {chunk_idx:02d}: Processed {total_processed:,}/1,732,544 ({speed:,.0f} entities/s) [{batch_time:.1f}s] | Matches: {total_matches_emitted:,}")
    gc.collect()

total_time = time.perf_counter() - t_start
match_ratio = (total_matches_emitted / total_processed) * 100
print(f"
🎉 Test Inference Complete! Processed {total_processed:,} entities in {total_time/60:.2f} minutes.")
print(f"📊 Final Match Profile: {total_matches_emitted:,} matched entities ({match_ratio:.2f}%) | {total_processed - total_matches_emitted:,} singletons ({100 - match_ratio:.2f}%)")


🚀 Streaming test_source1.tsv from S3 in batches of 150,000 ...
  • Batch 01: Processed 150,000/1,732,544 (1,519 entities/s) [98.8s] | Matches: 78,586
  • Batch 02: Processed 300,000/1,732,544 (1,579 entities/s) [95.0s] | Matches: 156,914
  • Batch 03: Processed 450,000/1,732,544 (1,568 entities/s) [95.6s] | Matches: 235,534
  • Batch 04: Processed 600,000/1,732,544 (1,532 entities/s) [97.9s] | Matches: 313,911
  • Batch 05: Processed 750,000/1,732,544 (1,596 entities/s) [94.0s] | Matches: 392,642
  • Batch 06: Processed 900,000/1,732,544 (1,653 entities/s) [90.7s] | Matches: 471,144
  • Batch 07: Processed 1,050,000/1,732,544 (1,646 entities/s) [91.1s] | Matches: 549,170
  • Batch 08: Processed 1,200,000/1,732,544 (1,580 entities/s) [94.9s] | Matches: 627,615
  • Batch 09: Processed 1,350,000/1,732,544 (1,598 entities/s) [93.9s] | Matches: 705,896
  • Batch 10: Processed 1,500,000/1,732,544 (1,611 entities/s) [93.1s] | Matches: 784,248
  • Batch 11: Processed 1,650,000/1,732,544 (1,641

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  • Batch 12: Processed 1,732,544/1,732,544 (1,672 entities/s) [49.4s] | Matches: 905,225

🎉 Test Inference Complete! Processed 1,732,544 entities in 18.91 minutes.
📊 Final Match Profile: 905,225 matched entities (52.25%) | 827,319 singletons (47.75%)


## Run Official Submission Validator Against Strict Rules

In [9]:
import os
import subprocess

# 1. Locate validate_submission.py in Colab files
validator_script = None
possible_locations = [
    "validate_submission.py",
    "utils/validate_submission.py",
    "/content/validate_submission.py",
    "/content/utils/validate_submission.py"
]

for loc in possible_locations:
    if os.path.exists(loc):
        validator_script = loc
        break

if not validator_script:
    raise FileNotFoundError(
        "Could not find validate_submission.py! Please upload it into the Colab file explorer on the left."
    )

print(f"✅ Found validator at: {validator_script}")

# 2. Ensure test_source1.tsv is locally present for required ID verification
test_dir = "dataset/test"
os.makedirs(test_dir, exist_ok=True)
test_s1_local = os.path.join(test_dir, "test_source1.tsv")

if not os.path.exists(test_s1_local):
    print("⏳ Downloading test_source1.tsv from S3 for ground validation check...")
    fs.get(f"{TEST_BASE_S3}/test_source1.tsv", test_s1_local)
    print("✅ test_source1.tsv downloaded.")

# 3. Execute the official validator
print("\n" + "=" * 80)
print(f"🔍 EXECUTING OFFICIAL VALIDATOR: python3 {validator_script}")
print("=" * 80)

val_res = subprocess.run([
    "python3", validator_script,
    "--matching", MATCHING_TSV,
    "--candidate", CANDIDATE_TSV,
    "--test-dir", test_dir
], capture_output=True, text=True)

print(val_res.stdout)
if val_res.stderr:
    print("STDERR:\n", val_res.stderr)

if val_res.returncode == 0:
    print("\n🏆 VALIDATION PASSED (Exit Code: 0)!")
    print("Both TSV files strictly conform to the scorer specifications and are 100% safe to submit.")
else:
    print(f"\n❌ VALIDATION FAILED (Exit Code: {val_res.returncode}). Please review the errors above.")

✅ Found validator at: validate_submission.py

🔍 EXECUTING OFFICIAL VALIDATOR: python3 validate_submission.py
ML Challenge 2026 — submission validator
  test dir: dataset/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (827319 empty, 905225 non-empty).
  candidate_pairs.tsv: 1732544 rows (7395 empty, 1725149 non-empty).

PASS — no blocking issues found. Safe to submit.


🏆 VALIDATION PASSED (Exit Code: 0)!
Both TSV files strictly conform to the scorer specifications and are 100% safe to submit.


## Upload Validated Files to Central S3 & Create final_submission.zip

In [10]:
# 1. Upload both TSVs to S3 submissions folder
print(f"⏳ Uploading matching_results.tsv to {SUBMISSIONS_S3}/matching_results.tsv ...")
fs.put(MATCHING_TSV, f"{S3_BUCKET}/submissions/matching_results.tsv")

print(f"⏳ Uploading candidate_pairs.tsv  to {SUBMISSIONS_S3}/candidate_pairs.tsv ...")
fs.put(CANDIDATE_TSV, f"{S3_BUCKET}/submissions/candidate_pairs.tsv")

# 2. Package into official final_submission.zip
print("\n📦 Packaging final submission archive...")
!zip -q -r final_submission.zip output/

zip_size_mb = os.path.getsize("final_submission.zip") / (1024 * 1024)
print(f"✅ Created final_submission.zip ({zip_size_mb:.2f} MB)")
print("🚀 Ready for final leaderboard upload!")

⏳ Uploading matching_results.tsv to s3://amz-ml-crazy-dave-bucket-177683310295-us-east-1-an/submissions/matching_results.tsv ...
⏳ Uploading candidate_pairs.tsv  to s3://amz-ml-crazy-dave-bucket-177683310295-us-east-1-an/submissions/candidate_pairs.tsv ...

📦 Packaging final submission archive...
✅ Created final_submission.zip (73.56 MB)
🚀 Ready for final leaderboard upload!


In [11]:
from google.colab import files

print("⬇️ Downloading matching_results.tsv for Unstop Leaderboard...")
files.download("output/matching_results.tsv")

⬇️ Downloading matching_results.tsv for Unstop Leaderboard...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>